# Lounge Eligibility Access 

## 1. Load the Data & Explore Key Fields

Load the BA summer schedule dataset and check which fields are stable, reusable
categories (`HAUL`, `TIME_OF_DAY`, `ARRIVAL_REGION`) versus fields that won't help
with grouping.

In [1]:
import pandas as pd
df = pd.read_excel("British Airways Summer Schedule Dataset - Forage Data Science Task 1.xlsx")
print(df.dtypes)
print (df.HAUL.unique())          # → ['LONG', 'SHORT']
print (df.TIME_OF_DAY.unique())   # → ['Morning','Lunchtime','Afternoon','Evening']
print (df.ARRIVAL_REGION.unique())# → ['North America','Europe','Asia','Middle East']

FLIGHT_DATE             datetime64[ns]
FLIGHT_TIME                     object
TIME_OF_DAY                     object
AIRLINE_CD                      object
FLIGHT_NO                       object
DEPARTURE_STATION_CD            object
ARRIVAL_STATION_CD              object
ARRIVAL_COUNTRY                 object
ARRIVAL_REGION                  object
HAUL                            object
AIRCRAFT_TYPE                   object
FIRST_CLASS_SEATS                int64
BUSINESS_CLASS_SEATS             int64
ECONOMY_SEATS                    int64
TIER1_ELIGIBLE_PAX               int64
TIER2_ELIGIBLE_PAX               int64
TIER3_ELIGIBLE_PAX               int64
dtype: object
['LONG' 'SHORT']
['Afternoon' 'Morning' 'Evening' 'Lunchtime']
['North America' 'Europe' 'Asia' 'Middle East']


## 2. Check How Haul and Region Relate

Before building a grouping scheme, confirm whether `HAUL` and `ARRIVAL_REGION` are
independent or overlapping — this affects how many real categories exist.

In [2]:
pd.crosstab(df.HAUL, df.ARRIVAL_REGION)

ARRIVAL_REGION,Asia,Europe,Middle East,North America
HAUL,,,,
LONG,679,0,688,2658
SHORT,0,5975,0,0


## 3. Compare Cabin Configuration by Haul/Region

Check average First/Business/Economy seat counts per group — this reveals whether
short-haul and long-haul flights differ meaningfully in cabin mix.

In [3]:
df.groupby(['HAUL','ARRIVAL_REGION'])[['FIRST_CLASS_SEATS','BUSINESS_CLASS_SEATS','ECONOMY_SEATS']].mean()

FIRST_CLASS_SEATS  BUSINESS_CLASS_SEATS  ECONOMY_SEATS
HAUL  ARRIVAL_REGION                                                        
LONG  Asia                     3.793814             47.228277     239.633284
      Middle East              3.793605             46.853198     240.991279
      North America            3.869827             47.595561     240.696764
SHORT Europe                   0.000000             10.029456     169.970544

## 4. Sanity-Check the Dataset's Predefined Tier Columns

The dataset includes `TIER1/2/3_ELIGIBLE_PAX` columns. Before using them, verify
they actually reflect real cabin configuration — e.g., does `TIER1_ELIGIBLE_PAX`
correlate with `FIRST_CLASS_SEATS` as it logically should?

In [4]:
df[['FIRST_CLASS_SEATS','TIER1_ELIGIBLE_PAX']].corr()  # → ~0 correlation

,FIRST_CLASS_SEATS,TIER1_ELIGIBLE_PAX
FIRST_CLASS_SEATS,1.000000,-0.008837
TIER1_ELIGIBLE_PAX,-0.008837,1.000000


## 5. Build the Flight Grouping (With Region)

Define reusable, fleet-independent groups using `HAUL` × `ARRIVAL_REGION` ×
time-of-day bucket (Morning vs. Afternoon/Evening).

In [5]:
# Step 1: define your time-of-day bucket
# TIME_OF_DAY currently has 4 values: Morning, Lunchtime, Afternoon, Evening
# Decide how you want to collapse these into your 2 buckets (AM vs PM)
def time_bucket(tod):
    if tod == "Morning":
        return "Morning"
    else:
        return "Afternoon/Evening"   # Lunchtime, Afternoon, Evening all become PM

df["TIME_BUCKET"] = df["TIME_OF_DAY"].apply(time_bucket)

# Step 2: build the region label — for long-haul, keep the actual region;
# for short-haul, everything is Europe anyway
df["REGION_GROUP"] = df["ARRIVAL_REGION"]  # already clean: Europe/NA/Asia/Middle East

# Step 3: combine into your final grouping
df["GROUP"] = df["HAUL"].str.title() + "-haul " + df["REGION_GROUP"] + " – " + df["TIME_BUCKET"]

print(df["GROUP"].value_counts())


GROUP
Short-haul Europe – Afternoon/Evening          3906
Short-haul Europe – Morning                    2069
Long-haul North America – Afternoon/Evening    1692
Long-haul North America – Morning               966
Long-haul Middle East – Afternoon/Evening       439
Long-haul Asia – Afternoon/Evening              433
Long-haul Middle East – Morning                 249
Long-haul Asia – Morning                        246
Name: count, dtype: int64


## 6. Check Aircraft Mix as Supporting Evidence

Aircraft type isn't used as a grouping column (the task requires fleet-independent
groups), but checking it here shows *why* regions differ — e.g., which groups fly
more First-Class-equipped aircraft — which justifies the assumption percentages later.

In [6]:
df["TOTAL_SEATS"] = df.FIRST_CLASS_SEATS + df.BUSINESS_CLASS_SEATS + df.ECONOMY_SEATS

# 1. What aircraft types actually show up in each group?
aircraft_mix = pd.crosstab(df["GROUP"], df["AIRCRAFT_TYPE"])
print("=== Aircraft type mix per group (counts) ===")
print(aircraft_mix)
print()

# 2. Same thing as a percentage breakdown (more useful for justification)
aircraft_mix_pct = aircraft_mix.div(aircraft_mix.sum(axis=1), axis=0).round(3) * 100
print("=== Aircraft type mix per group (%) ===")
print(aircraft_mix_pct)
print()

# 3. Average seat config per group (this reflects the aircraft mix automatically)
seat_avg = df.groupby("GROUP")[["FIRST_CLASS_SEATS","BUSINESS_CLASS_SEATS","ECONOMY_SEATS","TOTAL_SEATS"]].mean().round(1)
print("=== Average seat configuration per group ===")
print(seat_avg)

=== Aircraft type mix per group (counts) ===
AIRCRAFT_TYPE                                A320  A350  A380  B777  B787
GROUP                                                                    
Long-haul Asia – Afternoon/Evening              0    58    40   196   139
Long-haul Asia – Morning                        0    29    24   116    77
Long-haul Middle East – Afternoon/Evening       0    57    40   214   128
Long-haul Middle East – Morning                 0    28    19   133    69
Long-haul North America – Afternoon/Evening     0   235   161   769   527
Long-haul North America – Morning               0   128    92   450   296
Short-haul Europe – Afternoon/Evening        3906     0     0     0     0
Short-haul Europe – Morning                  2069     0     0     0     0

=== Aircraft type mix per group (%) ===
AIRCRAFT_TYPE                                 A320  A350  A380  B777  B787
GROUP                                                                     
Long-haul Asia – Afterno

## 7. Lookup Table A — Built From the Predefined Tier Columns (With Region)

Average the dataset's given `TIER1/2/3_ELIGIBLE_PAX` percentages within each group,
and check the within-group standard deviation to see how reliable those averages are.

In [7]:
# Turn the predefined pax counts into percentages of total seats, per flight
df["TIER1_PCT"] = df["TIER1_ELIGIBLE_PAX"] / df["TOTAL_SEATS"]
df["TIER2_PCT"] = df["TIER2_ELIGIBLE_PAX"] / df["TOTAL_SEATS"]
df["TIER3_PCT"] = df["TIER3_ELIGIBLE_PAX"] / df["TOTAL_SEATS"]

# Average those percentages within each group to build the lookup table
lookup_from_predefined = df.groupby("GROUP")[["TIER1_PCT","TIER2_PCT","TIER3_PCT"]].mean().round(4)

print("=== Lookup table built from PREDEFINED tier data ===")
print(lookup_from_predefined)

# Optional: also check the row-level spread (std dev) within each group —
# a high std relative to the mean is a red flag that the "average" isn't
# a stable, trustworthy number for that group
lookup_std = df.groupby("GROUP")[["TIER1_PCT","TIER2_PCT","TIER3_PCT"]].std().round(4)
print()
print("=== Std dev within each group (how noisy is the given data?) ===")
print(lookup_std)

=== Lookup table built from PREDEFINED tier data ===
                                             TIER1_PCT  TIER2_PCT  TIER3_PCT
GROUP                                                                       
Long-haul Asia – Afternoon/Evening              0.0020     0.0291     0.1113
Long-haul Asia – Morning                        0.0023     0.0278     0.1072
Long-haul Middle East – Afternoon/Evening       0.0022     0.0279     0.1071
Long-haul Middle East – Morning                 0.0021     0.0296     0.1123
Long-haul North America – Afternoon/Evening     0.0022     0.0292     0.1114
Long-haul North America – Morning               0.0022     0.0300     0.1140
Short-haul Europe – Afternoon/Evening           0.0035     0.0442     0.1691
Short-haul Europe – Morning                     0.0034     0.0436     0.1674

=== Std dev within each group (how noisy is the given data?) ===
                                             TIER1_PCT  TIER2_PCT  TIER3_PCT
GROUP                             

## 8. Lookup Table B — Predefined Tier Columns, Region Dropped

Rebuild the grouping using only `HAUL` + time-of-day (no region), to compare a
simpler 4-group version against the 8-group version above.

In [8]:
# GROUP now uses only HAUL + TIME_BUCKET — region is dropped
df["GROUP_NO_REGION"] = df["HAUL"].str.title() + "-haul - " + df["TIME_BUCKET"]

df["TOTAL_SEATS"] = df.FIRST_CLASS_SEATS + df.BUSINESS_CLASS_SEATS + df.ECONOMY_SEATS

print("Flights per group (no region):")
print(df["GROUP_NO_REGION"].value_counts())
print()

# Build lookup from the PREDEFINED tier columns, same method as before
df["TIER1_PCT"] = df["TIER1_ELIGIBLE_PAX"] / df["TOTAL_SEATS"]
df["TIER2_PCT"] = df["TIER2_ELIGIBLE_PAX"] / df["TOTAL_SEATS"]
df["TIER3_PCT"] = df["TIER3_ELIGIBLE_PAX"] / df["TOTAL_SEATS"]

lookup_no_region_predefined = df.groupby("GROUP_NO_REGION")[["TIER1_PCT","TIER2_PCT","TIER3_PCT"]].mean().round(4)
print("=== Lookup table (Haul + Time only) — from PREDEFINED data ===")
print(lookup_no_region_predefined)

Flights per group (no region):
GROUP_NO_REGION
Short-haul - Afternoon/Evening    3906
Long-haul - Afternoon/Evening     2564
Short-haul - Morning              2069
Long-haul - Morning               1461
Name: count, dtype: int64

=== Lookup table (Haul + Time only) — from PREDEFINED data ===
                                TIER1_PCT  TIER2_PCT  TIER3_PCT
GROUP_NO_REGION                                                
Long-haul - Afternoon/Evening      0.0021     0.0290     0.1106
Long-haul - Morning                0.0022     0.0295     0.1125
Short-haul - Afternoon/Evening     0.0035     0.0442     0.1691
Short-haul - Morning               0.0034     0.0436     0.1674


## 9. Lookup Table C — My Own Assumptions (Region Dropped)

Independent, justified percentage estimates for the simplified 4-group (no-region)
scheme, based on business vs. leisure passenger mix reasoning.

In [9]:
tier_assumptions_no_region = {
    "Short-haul - Morning":            {"TIER1_PCT": 0.00, "TIER2_PCT": 0.05, "TIER3_PCT": 0.20},
    "Short-haul - Afternoon/Evening":  {"TIER1_PCT": 0.00, "TIER2_PCT": 0.03, "TIER3_PCT": 0.15},
    "Long-haul - Morning":             {"TIER1_PCT": 0.02, "TIER2_PCT": 0.08, "TIER3_PCT": 0.26},
    "Long-haul - Afternoon/Evening":  {"TIER1_PCT": 0.01, "TIER2_PCT": 0.06, "TIER3_PCT": 0.22},
}
lookup_no_region_assumption = pd.DataFrame(tier_assumptions_no_region).T

print("=== WITHOUT region (4 groups) ===")
print(lookup_no_region_assumption)

=== WITHOUT region (4 groups) ===
                                TIER1_PCT  TIER2_PCT  TIER3_PCT
Short-haul - Morning                 0.00       0.05       0.20
Short-haul - Afternoon/Evening       0.00       0.03       0.15
Long-haul - Morning                  0.02       0.08       0.26
Long-haul - Afternoon/Evening        0.01       0.06       0.22


## 10. Lookup Table D — My Own Assumptions (With Region) + Passenger Estimates

Final assumption-based percentages for the 8-group (with-region) scheme, converted
into estimated eligible passengers per flight and per group by multiplying against
total seat counts.

In [10]:
df["TIME_BUCKET"] = df["TIME_OF_DAY"].apply(time_bucket)
df["GROUP"] = df["HAUL"].str.title() + "-haul " + df["ARRIVAL_REGION"] + " - " + df["TIME_BUCKET"]
df["TOTAL_SEATS"] = df.FIRST_CLASS_SEATS + df.BUSINESS_CLASS_SEATS + df.ECONOMY_SEATS

# Your assumption-based percentages (1 decimal, e.g. 3.0% not 0.03)
tier_assumptions = {
    "Short-haul Europe - Morning":                 {"TIER1_PCT1": 0.0, "TIER2_PCT1": 5.0, "TIER3_PCT1": 20.0},
    "Short-haul Europe - Afternoon/Evening":       {"TIER1_PCT1": 0.0, "TIER2_PCT1": 3.0, "TIER3_PCT1": 15.0},
    "Long-haul North America - Morning":           {"TIER1_PCT1": 3.0, "TIER2_PCT1": 10.0, "TIER3_PCT1": 30.0},
    "Long-haul North America - Afternoon/Evening": {"TIER1_PCT1": 2.0, "TIER2_PCT1": 8.0, "TIER3_PCT1": 25.0},
    "Long-haul Middle East - Morning":             {"TIER1_PCT1": 2.0, "TIER2_PCT1": 7.0, "TIER3_PCT1": 22.0},
    "Long-haul Middle East - Afternoon/Evening":   {"TIER1_PCT1": 1.0, "TIER2_PCT1": 6.0, "TIER3_PCT1": 20.0},
    "Long-haul Asia - Morning":                    {"TIER1_PCT1": 1.0, "TIER2_PCT1": 5.0, "TIER3_PCT1": 20.0},
    "Long-haul Asia - Afternoon/Evening":          {"TIER1_PCT1": 1.0, "TIER2_PCT1": 4.0, "TIER3_PCT1": 18.0},
}
lookup = pd.DataFrame(tier_assumptions).T.reset_index().rename(columns={"index": "GROUP"})

# Merge percentages onto flight-level data
df = df.merge(lookup, on="GROUP", how="left")

# Multiply: percentage (as fraction) x total seats = estimated eligible passengers
df["EST_TIER1_PAX"] = (df["TOTAL_SEATS"] * df["TIER1_PCT1"] / 100).round(1)
df["EST_TIER2_PAX"] = (df["TOTAL_SEATS"] * df["TIER2_PCT1"] / 100).round(1)
df["EST_TIER3_PAX"] = (df["TOTAL_SEATS"] * df["TIER3_PCT1"] / 100).round(1)

# Per-flight average estimated pax, grouped — this is your "typical day" planning number
summary = df.groupby("GROUP")[["EST_TIER1_PAX", "EST_TIER2_PAX", "EST_TIER3_PAX"]].mean().round(1)
print("=== Average estimated eligible passengers per flight, by group ===")
print(summary)

# Total estimated pax across the whole sample (a "how many people, in total" figure)
totals = df.groupby("GROUP")[["EST_TIER1_PAX", "EST_TIER2_PAX", "EST_TIER3_PAX"]].sum().round(1)
print()
print("=== Total estimated eligible passengers across all flights in each group ===")
print(totals)

lookup_no_region = df.groupby("GROUP_NO_REGION")[["EST_TIER1_PAX", "EST_TIER2_PAX", "EST_TIER3_PAX"]].mean().round(1)
print("=== Lookup table (Haul + Time only) — from Assumption data ===")
print(lookup_no_region)

=== Average estimated eligible passengers per flight, by group ===
                                             EST_TIER1_PAX  EST_TIER2_PAX  \
GROUP                                                                       
Long-haul Asia - Afternoon/Evening                     2.9           11.6   
Long-haul Asia - Morning                               2.9           14.6   
Long-haul Middle East - Afternoon/Evening              2.9           17.5   
Long-haul Middle East - Morning                        5.8           20.3   
Long-haul North America - Afternoon/Evening            5.8           23.4   
Long-haul North America - Morning                      8.8           29.3   
Short-haul Europe - Afternoon/Evening                  0.0            5.4   
Short-haul Europe - Morning                            0.0            9.0   

                                             EST_TIER3_PAX  
GROUP                                                       
Long-haul Asia - Afternoon/Evening      

## 11. Conclusion — Final Chosen Approach

**Final answer: with-region grouping (8 groups), assumption-based percentages.**

Region was kept because it's tied to a real difference in cabin/aircraft mix (see
Section 6) that a region-free model would blend away — exactly the signal BA needs
to decide where lounge investment should be prioritized. The predefined tier columns
(Section 4/7) were not used as the final answer because they failed a basic
logic check: short-haul flights, which carry no First Class cabin, showed *higher*
average Tier 1 eligibility than long-haul flights that do have First cabins.

## 12. Export Final Lookup Table

Save the chosen lookup table (Section 10) to a clean file for reuse outside this notebook.

In [11]:
final_lookup = pd.DataFrame(tier_assumptions).T.reset_index().rename(columns={"index": "GROUP"})
final_lookup.to_csv("final_lounge_lookup_table.csv", index=False)
final_lookup

,GROUP,TIER1_PCT1,TIER2_PCT1,TIER3_PCT1
0,Short-haul Europe - Morning,0.0,5.0,20.0
1,Short-haul Europe - Afternoon/Evening,0.0,3.0,15.0
2,Long-haul North America - Morning,3.0,10.0,30.0
3,Long-haul North America - Afternoon/Evening,2.0,8.0,25.0
4,Long-haul Middle East - Morning,2.0,7.0,22.0
5,Long-haul Middle East - Afternoon/Evening,1.0,6.0,20.0
6,Long-haul Asia - Morning,1.0,5.0,20.0
7,Long-haul Asia - Afternoon/Evening,1.0,4.0,18.0
